In [ ]:
###PART 2

############################################################################
#Zwischenstand anschauen
############################################################################

#Trainingsergebnis zwischenprüfen

In [ ]:
#Speicher überprüfen und nach Training auswerten

import torch
import gc

def show_mem(tag=""):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        print(f"\n--- {tag} ---")
        print(f"allocated:      {torch.cuda.memory_allocated()/1024**3:.2f} GB")
        print(f"reserved:       {torch.cuda.memory_reserved()/1024**3:.2f} GB")
        print(f"max allocated:  {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
        print(f"max reserved:   {torch.cuda.max_memory_reserved()/1024**3:.2f} GB")
        print(torch.cuda.memory_summary())
        
torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

torch.cuda.memory._record_memory_history()

show_mem("start")

In [ ]:
#Inferenzen Generieren zum Test der trainierten Modelle

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

model_path = r"out_pilot_##3b-850/checkpoint-850" #3B-Instruct #3b-hf
adapter_path = r"out_pilot_##3b/checkpoint-1000"  #3b-instruct #3b ##3b

assert torch.cuda.is_available(), #cuda test

tok = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

def generate_only_new(model, prompt: str, max_new_tokens=80, temperature=0.7):
    #Generiert nur neue Tokens
    model.eval()
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=60,
            repetition_penalty=1.1,
            do_sample=False,
            temperature=temperature,
        )
    cut = inputs["input_ids"].shape[-1]
    gen_ids = out[0][cut:]
    return tok.decode(gen_ids, skip_special_tokens=True).strip()

def free_gpu(*objs):
    #GPU Ram freigeben
    for o in objs:
        try:
            del o
        except:
            pass
    torch.cuda.empty_cache()


In [ ]:
#Prompt vorbereiten und Modell laden
prompt = "User: Hey hier ist Luminara, hast du nächstes we Zeit? :)\nAssistant:"

# Base Modell direkt auf GPU laden 
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cuda",
    dtype=torch.float16,
    local_files_only=True
)

base_answer = generate_only_new(base_model, prompt)
print("=== BASE ===")
print(base_answer)

del base_model
import gc
gc.collect()
torch.cuda.empty_cache()

show_mem("after deleting base model")

In [ ]:
#Base auf CPU laden 
base_cpu = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map={"": "cpu"},
    dtype=torch.float16,
    local_files_only=True
)

ft = PeftModel.from_pretrained(base_cpu, adapter_path)
ft_merged = ft.merge_and_unload().eval()

#Umladen auf GPU, um Fehler zu vermeiden
ft_merged = ft_merged.to("cuda")

ft_answer = generate_only_new(ft_merged, prompt)
print("=== FINE-TUNED ===")
print(ft_answer)

del base_cpu, ft, ft_merged
import gc
gc.collect()
torch.cuda.empty_cache()

show_mem("after deleting ft model")

In [ ]:
####Weitere Prompt Beispiele###

In [ ]:
test_prompts = [
    "User: Ey sorry hab gestern voll vercheckt zu antworten 😅\nAssistant:",
    "User: Alles gut bei dir heute?\nAssistant:",
    "User: Haha ja stimmt, war echt lustig gestern\nAssistant:",
    "User: Ich weiß noch nicht, ob ich Zeit hab 🤔\nAssistant:",
    "User: Danke dir fürs Bescheid sagen!\nAssistant:",
    "User: Bin grad mega gestresst wegen Uni 😩\nAssistant:",
    "User: Wollen wir das morgen klären?\nAssistant:",
    "User: Ich glaub das wird knapp heute\nAssistant:",
    "User: Ey das war echt nicht cool von dir…\nAssistant:",
    "User: Gute Nacht 😊\nAssistant:",
]


In [ ]:
#Neue Instanz des Modells laden

def generate_only_new(model, prompt, max_new_tokens=80, temperature=0.7):
    model.eval()
    inputs = tok(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            eos_token_id=tok.eos_token_id,
            pad_token_id=tok.eos_token_id,
        )

    cut = inputs["input_ids"].shape[-1]
    gen_ids = out[0][cut:]
    return tok.decode(gen_ids, skip_special_tokens=True).strip()


In [ ]:
#10 Inferenzen pro Modell generieren

results = []

#---------- BASE ----------
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="cuda",
    offload_folder="offload",
    dtype=torch.float16,
    local_files_only=True
)

for p in test_prompts:
    base_out = generate_only_new(base_model, p)
    results.append({
        "prompt": p,
        "base": base_out,
        "ft": None
    })

del base_model
gc.collect()
torch.cuda.empty_cache()

#---------- FINE-TUNED ----------
base_cpu = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map={"": "cpu"},
    dtype=torch.float16,
    local_files_only=True
)

ft = PeftModel.from_pretrained(base_cpu, adapter_path)
ft_merged = ft.merge_and_unload().eval().to("cuda")

for i, p in enumerate(test_prompts):
    ft_out = generate_only_new(ft_merged, p)
    results[i]["ft"] = ft_out

del base_cpu, ft, ft_merged
gc.collect()
torch.cuda.empty_cache()


In [ ]:
show_mem("after model load 3")

In [ ]:
#torch.cuda.memory._dump_snapshot("cuda_snapshot.pickle")

In [ ]:
torch.cuda.empty_cache()

In [ ]:
import gc
import torch

cuda_tensors = []
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            cuda_tensors.append((type(obj), tuple(obj.shape), obj.dtype))
    except:
        pass

print(f"CUDA tensors alive: {len(cuda_tensors)}")
for t in cuda_tensors[:20]:
    print(t)

In [ ]:
gc.collect()

In [ ]:
for i, r in enumerate(results):
    print(f"\n### Beispiel {i+1}")
    print("PROMPT:", r["prompt"])
    print("BASE :", r["base"])
    print("FT   :", r["ft"])


In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("base_vs_ft_examples.csv", index=False)
